In [ ]:
import dataclasses as dc
import json
import os

import torch
from torchvision import datasets, transforms

from mnist_nets import Net, data_path
from mnist_vae import VAE

import matplotlib.pyplot as plt

In [ ]:
cnn_args_path = "checkpoints/440533/mnist_args_2d.json"
vae_args_path = "checkpoints/443369/mnist_args_32d.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert os.path.isfile(cnn_args_path)
with open(cnn_args_path) as f:
    args = json.load(f)

args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)

checkpoints_folder = '/'.join(cnn_args_path.split('/')[:-1])
checkpoint_path = os.path.join(checkpoints_folder, f"mnist_hyp_cnn_{args.best_epoch_hyp}_{args.dim}d.pt")
assert os.path.isfile(checkpoint_path)

cnn = Net(args).to(device)
cnn.load_state_dict(torch.load(checkpoint_path, map_location=device))

assert os.path.isfile(vae_args_path)
with open(vae_args_path) as f:
    args = json.load(f)

args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)

checkpoints_folder = '/'.join(vae_args_path.split('/')[:-1])
checkpoint_path = os.path.join(checkpoints_folder, f"mnist_vae_{args.best_epoch}_{args.dim}d.pt")
assert os.path.isfile(checkpoint_path)

vae = VAE(args).to(device)
vae.load_state_dict(torch.load(checkpoint_path, map_location=device))

torch.manual_seed(args.seed)

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True}
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=False,
        transform=transforms.Compose([
            transforms.ToTensor(), 
        ]),
    ),
    batch_size=32,
    shuffle=True,
    **kwargs
)

In [ ]:
x = next(iter(test_loader))[0].to(device)
vae.eval()
with torch.no_grad():
    recon, *_ = vae(x)

fig, ax = plt.subplots(1, 2)

ax[0].imshow(x[0, 0].cpu().numpy(), cmap="gray")
ax[1].imshow(recon[0, 0].cpu().numpy(), cmap="gray")

ax[0].set_title("Original")
ax[1].set_title("Reconstruction")

ax[0].axis("off")
ax[1].axis("off")

fig.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets

x = next(iter(test_loader))[0].to(device)
vae.eval()
with torch.no_grad():
    latent = vae.reparam(*vae.encode(x))

z1 = latent[0]
x1 = vae.decode(z1).detach().numpy()
x1_pc = cnn.embed(vae.decode(z1)).detach().numpy()[0]

z2 = latent[1]
x2 = vae.decode(z2).detach().numpy()
x2_pc = cnn.embed(vae.decode(z2)).detach().numpy()[0]

lerp_slider = widgets.FloatSlider(
    value=0.5,
    min=0,
    max=1,
    description="alpha",
    continuous_update=True,
)

def _lerp_slider(alpha):
    fig_nums, ax_nums = plt.subplots(1, 3)
    
    z3 = (1-alpha) * z1 + alpha * z2
    x3 = vae.decode(z3).detach().numpy()
    x3_pc = cnn.embed(vae.decode(z3)).detach().numpy()[0]

    ax_nums[0].set_title("Image 1")
    ax_nums[1].set_title("Image 2")
    ax_nums[2].set_title("Lerp")

    for i, im in enumerate([x1, x2, x3]):
        ax_nums[i].imshow(im[0, 0], cmap="gray")    
        ax_nums[i].axis("off")

    fig_nums.tight_layout()
    display(fig_nums)
    plt.close(fig_nums)

    fig_pc, ax_pc = plt.subplots(figsize=(6, 6))
    ax_pc.add_artist(plt.Circle((0, 0), 1, color="black", fill=False))
    ax_pc.scatter(x1_pc[0], x1_pc[1], label="x1")
    ax_pc.scatter(x2_pc[0], x2_pc[1], label="x2")
    ax_pc.scatter(x3_pc[0], x3_pc[1], label="lerp")

    fig_pc.tight_layout()
    ax_pc.set_xlim(-1.1, 1.1)
    ax_pc.set_ylim(-1.1, 1.1)
    ax_pc.set_aspect("equal")

    ax_pc.legend()
    display(fig_pc)
    plt.close(fig_pc)
    
plot_output = widgets.interactive_output(
    _lerp_slider, {"alpha": lerp_slider}
)

widgets.VBox([lerp_slider, plot_output])

In [ ]:
import numpy as np
from matplotlib import cm

x = next(iter(test_loader))[0].to(device)
vae.eval()
with torch.no_grad():
    latent = vae.reparam(*vae.encode(x))

z1 = latent[0]
x1 = vae.decode(z1).detach().numpy()
x1_pc = cnn.embed(vae.decode(z1)).detach().numpy()[0]

z2 = latent[1]
x2 = vae.decode(z2).detach().numpy()
x2_pc = cnn.embed(vae.decode(z2)).detach().numpy()[0]

num_interps = 64
interps = np.array([
    cnn.embed(vae.decode((1-alpha) * z1 + alpha * z2)).detach().numpy().reshape(2)
    for alpha in np.linspace(0, 1, num_interps)
])

fig, ax = plt.subplots(figsize=(6, 6))

cmap = cm.spring(np.linspace(0,1,num_interps))
ax.add_artist(plt.Circle((0, 0), 1, color="black", fill=False))
ax.scatter(interps[:,0], interps[:,1], c=cmap, label="interp", s=10)

for i in range(num_interps-1):
    ax.plot(interps[i:i+2,0], interps[i:i+2,1], c=cmap[i])

sm = plt.cm.ScalarMappable(cmap=cm.spring)
plt.colorbar(sm)

ax.legend()
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")
plt.show()

In [ ]:
def plot_hyp(vae, cnn, loader):
    vae.eval()
    cnn.eval()

    with torch.no_grad():
        for (x, y) in loader:
            x = x.to(device)
            recon = vae(x)[0]
            z = cnn.embed(recon).detach().numpy()
            
            plt.scatter(z[:, 0], z[:, 1], c=y, cmap="tab10", alpha=0.5)

    plt.tight_layout()
    plt.axis("equal")
    plt.show()

In [ ]:
plot_hyp(vae, cnn, test_loader)